# Part 2.3 — Grounding audit: is the prose supported by the evidence?

**Evidence base.** The CLI does not use the `equity_agents` team: every section is written by
`text_generator_agents.generate_text_section`, whose user prompt (`_prepare_user_prompt`) contains three tables:
the financial metrics & forecast table, peer EBITDA and peer EV/EBITDA. News goes to `news_summary` only. That prompt,
rebuilt here with the pipeline's own function from the CSVs Step 1 saved, is the entire evidence a section can be
grounded in.

**Method** (`audit/grounding.py`)
1. An LLM judge (`gpt-4.1`, temperature 0, structured output) splits each section into atomic claims and labels each
   *Supported / Unsupported / Contradicted* against that prompt, citing the cells it used and recording the year and
   status (actual vs forecast) the claim asserts.
2. Deterministic rules override the judge when the cited cells and the claim's wording conflict: a reported (A) value
   presented as a forecast, a forecast (E) value presented as reported, the historical CAGR attributed to a forecast
   period, or a year that matches none of the cited cells.
3. A number check looks up every $, % and x figure of each claim in the tables (and growth rates, CAGRs and ratios
   derived from them). It cannot judge wording, so it is a cross-check, not a verdict.
4. A sample of claims is reviewed by hand (`audit/grounding_review.csv`) to measure how often the judge is right.

**Texts.** For the five baseline runs: the seven LLM-written Step 1 sections (`analysis/*.txt`), plus the
`competitor_analysis` and `major_takeaways` the HTML report actually shows, which Step 2 regenerates with
`gpt-4o-mini`. `news_summary` is excluded: on the FMP free plan there is no news, so Step 1 writes a one-line
placeholder without calling the model.

Outputs: `audit/grounding_claims.csv` (every claim), `audit/grounding_rates.csv` (per section and company).

In [1]:
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "audit" else Path.cwd()
sys.path.insert(0, str(ROOT / "audit"))
import grounding as g
from openai import OpenAI
from modules.common_utils import load_config, get_api_key

pd.set_option("display.width", 250)
pd.set_option("display.max_colwidth", 140)
pd.set_option("display.max_rows", 200)

COMPANIES = {"NVDA": "NVIDIA Corporation", "AMD": "Advanced Micro Devices, Inc.", "INTC": "Intel Corporation",
             "AAPL": "Apple Inc.", "GOOGL": "Alphabet Inc."}
MODEL = "gpt-4.1"
cfg = load_config(str(g.SRC.parent / "config" / "config.ini"))
client = OpenAI(api_key=get_api_key(cfg, section="API_KEYS", key="openai_api_key"), max_retries=8)

results = [g.score_run(client, MODEL, t, name, skip=("news_summary",)) for t, name in COMPANIES.items()]
claims = pd.concat([c for c, _ in results], ignore_index=True)
sections = pd.concat([s for _, s in results], ignore_index=True)
claims.to_csv(ROOT / "audit" / "grounding_claims.csv", index=False)
print(f"{len(claims)} claims from {int((sections.status == 'scored').sum())} section texts; "
      f"judge calls served from cache: {bool(sections.get('cached', pd.Series(dtype=bool)).fillna(True).all())}")
claims["verdict"].value_counts()

438 claims from 45 section texts; judge calls served from cache: True


C:\Users\Ethan\AppData\Local\Temp\ipykernel_43348\2588881698.py:27: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  f"judge calls served from cache: {bool(sections.get('cached', pd.Series(dtype=bool)).fillna(True).all())}")


verdict
Supported       243
Unsupported     116
Contradicted     79
Name: count, dtype: int64

## What the writer is given

The complete prompt for NVIDIA's `risks` section — the system prompt asks for "5 key investment risks", the user
prompt is three tables. There is no line about products, customers, segments, R&D, net income, the share price,
regulation, supply chains or news.

In [2]:
sp, up = g.writer_prompt(g.OUT / "NVDA", "NVDA", COMPANIES["NVDA"], "risks")
print("SYSTEM:", sp, "\n")
print(up)

SYSTEM: You are a risk analyst. List 5 key investment risks in bullet point format. Be specific and concise. 

Company: NVIDIA Corporation (NVDA)

Financial Metrics:
|    | metrics             | 2022A             | 2023A              | 2024A             | 2025A             | 2026A             | 2025E          | 2026E          | 2027E           | CAGR   |
|---:|:--------------------|:------------------|:-------------------|:------------------|:------------------|:------------------|:---------------|:---------------|:----------------|:-------|
|  0 | Revenue             | 26914000000.0     | 26974000000.0      | 60922000000.0     | 130497000000.0    | 215938000000.0    | 226734900000.0 | 240338994000.0 | 249952553760.0  | 68.3%  |
|  1 | Cost of Operations  | 9439000000.0      | 11618000000.0      | 16621000000.0     | 32639000000.0     | 62475000000.0     | 63259037100.0  | 64651189386.0  | 64737711423.84  | nan    |
|  2 | SG&A                | 2166000000.0      | 2440000000.0       | 

## Grounding rate per section

Grounding rate = supported / total claims. *Contradicted* is the share of claims the evidence shows to be wrong;
*Unsupported* is the share that cannot have come from the evidence (prior knowledge or invention). Step 1 texts only;
the report versions are compared further down.

In [3]:
def rates(df, by):
    t = df.groupby(by)["verdict"].value_counts().unstack(fill_value=0)
    for v in ("Supported", "Unsupported", "Contradicted"):
        t[v] = t.get(v, 0)
    t["claims"] = t[["Supported", "Unsupported", "Contradicted"]].sum(axis=1)
    out = pd.DataFrame({"claims": t["claims"],
                        "grounding rate": t["Supported"] / t["claims"],
                        "unsupported": t["Unsupported"] / t["claims"],
                        "contradicted": t["Contradicted"] / t["claims"]})
    return out


step1 = claims[claims.variant == "step1"]
order = [s for s in g.SECTIONS if s in set(step1.section)]
by_section = rates(step1, "section").loc[order]
by_section.loc["ALL (step 1)"] = rates(step1.assign(all="ALL"), "all").iloc[0]
quant = rates(step1[step1.kind == "quantitative"], "section").reindex(order)
by_section["grounding rate, quantitative claims only"] = quant["grounding rate"]
by_section.loc["ALL (step 1)", "grounding rate, quantitative claims only"] = rates(
    step1[step1.kind == "quantitative"].assign(all="ALL"), "all").iloc[0]["grounding rate"]
by_section.style.format({c: "{:.0%}" for c in by_section.columns if c != "claims"} | {"claims": "{:.0f}"})

,claims,grounding rate,unsupported,contradicted,"grounding rate, quantitative claims only"
section,,,,,
tagline,25,64%,20%,16%,75%
company_overview,82,39%,55%,6%,87%
investment_overview,76,51%,26%,22%,69%
valuation_overview,54,57%,19%,24%,62%
risks,36,75%,6%,19%,79%
competitor_analysis,52,79%,12%,10%,84%
major_takeaways,20,50%,0%,50%,nan%
ALL (step 1),345,57%,26%,18%,75%


In [4]:
matrix = rates(step1, ["section", "ticker"])["grounding rate"].unstack().loc[order, list(COMPANIES)]
matrix["mean"] = matrix.mean(axis=1)
rates_out = rates(claims, ["variant", "section", "ticker"]).reset_index()
rates_out.to_csv(ROOT / "audit" / "grounding_rates.csv", index=False)
matrix.style.format("{:.0%}").background_gradient(cmap="RdYlGn", vmin=0, vmax=1)

ticker,NVDA,AMD,INTC,AAPL,GOOGL,mean
section,,,,,,
tagline,0%,75%,75%,80%,60%,58%
company_overview,24%,23%,50%,37%,58%,38%
investment_overview,36%,43%,71%,55%,56%,52%
valuation_overview,58%,44%,80%,64%,33%,56%
risks,33%,75%,80%,86%,100%,75%
competitor_analysis,73%,67%,100%,75%,78%,79%
major_takeaways,50%,25%,25%,50%,100%,50%


## The two sections the report actually shows: Step 1 vs Step 2

Step 1's `major_takeaways` was replaced by the canned fallback text in all five runs (the model's answer did not
contain the exact string `"Revenue Growth:"`), and Step 2 then regenerates both `competitor_analysis` and
`major_takeaways` with `gpt-4o-mini`. The report variant is what a reader sees.

In [5]:
variants = claims[claims.section.isin(["competitor_analysis", "major_takeaways"])]
cmp = rates(variants, ["section", "variant"])
cmp.style.format({c: "{:.0%}" for c in cmp.columns if c != "claims"} | {"claims": "{:.0f}"})

In [6]:
print("Step 1 major_takeaways (NVDA), i.e. the fallback text:\n")
print(sections.query("ticker == 'NVDA' and section == 'major_takeaways' and variant == 'step1'")["text"].iloc[0])

Step 1 major_takeaways (NVDA), i.e. the fallback text:

Revenue Growth: NVIDIA Corporation's revenue growth shows consistent performance trends.

Gross Profit Margin: NVIDIA Corporation's gross profit margins demonstrate operational effectiveness.

SG&A Expense Margin: NVIDIA Corporation's SG&A expense management shows disciplined cost control.

EBITDA Margin Stability: NVIDIA Corporation's EBITDA margin stability reflects strong underlying fundamentals.


## What kind of contradictions

Heuristic categories from the rule flags and the judge's reason, with every contradicted claim listed below.

In [7]:
def contradiction_type(r) -> str:
    f, reason, claim = r.rule_flags.lower(), r.reason.lower(), r.claim.lower()
    if "presented as a forecast" in f or (("actual" in reason or "reported" in reason)
                                           and any(w in reason for w in ("project", "forecast", "guidance", "expected"))):
        return "reported actual presented as forecast"
    if "presented as reported" in f:
        return "forecast presented as reported"
    if "cagr" in f or "cagr" in reason:
        return "CAGR attributed to wrong period"
    if any(w in claim for w in ("consensus", "guidance", "analyst")):
        return "invented source (consensus / guidance)"
    if "claim says" in f:
        return "wrong year"
    if any(w in reason for w in ("double", "each year", "consistent", "overstat", "exaggerat", "not show", "does not support")):
        return "quantifier / trend not supported by the numbers"
    return "wrong value or comparison"


contra = claims[claims.verdict == "Contradicted"].copy()
contra["type"] = contra.apply(contradiction_type, axis=1)
summary = contra.groupby(["type"]).size().sort_values(ascending=False).to_frame("claims")
summary["share"] = summary["claims"] / summary["claims"].sum()
summary.style.format({"share": "{:.0%}"})

,claims,share
type,,
reported actual presented as forecast,31,39%
CAGR attributed to wrong period,22,28%
wrong value or comparison,12,15%
quantifier / trend not supported by the numbers,11,14%
invented source (consensus / guidance),3,4%


In [8]:
contra.sort_values(["type", "ticker", "section"])[["type", "ticker", "section", "variant", "claim", "reason", "rule_flags"]]

,type,ticker,section,variant,claim,reason,rule_flags
278,CAGR attributed to wrong period,AAPL,company_overview,step1,A compound annual growth rate (CAGR) of 3.3% is projected through 2027.,The EVIDENCE lists a 3.3% CAGR for revenue from 2021A to 2027E.,historical CAGR attributed to a forecast period
289,CAGR attributed to wrong period,AAPL,investment_overview,step1,"Revenue is expected to rebound, with consensus estimates pointing to a CAGR of 3.3% through 2027 and projected revenue of $481.7 billion...","The 2027E revenue and 3.3% CAGR are the pipeline's own forecasts, not consensus estimates; attributing them to consensus is incorrect.",historical CAGR attributed to a forecast period
346,CAGR attributed to wrong period,AAPL,major_takeaways,report,Apple is expected to see a more robust compound annual growth rate (CAGR) of 3.3% from 2021 to 2027.,"The table shows a 3.3% CAGR for Revenue from 2021A to 2027E, matching the claim.",historical CAGR attributed to a forecast period
314,CAGR attributed to wrong period,AAPL,risks,step1,Apple's revenue growth is projected to be only modest through 2027 (CAGR of 3.3%).,The CAGR for Revenue from 2021A to 2027E is 3.3% as shown in the table.,historical CAGR attributed to a forecast period
264,CAGR attributed to wrong period,AAPL,tagline,step1,"Apple Inc. maintains robust financial health, delivering steady revenue growth (3.3% CAGR through 2027)",The 3.3% revenue CAGR from 2021A to 2027E is directly stated in the evidence.,historical CAGR attributed to a forecast period
305,CAGR attributed to wrong period,AAPL,valuation_overview,step1,"Apple’s revenue growth is expected to be modest, with a 3.3% CAGR through 2027.","The table shows a 3.3% revenue CAGR from 2021A to 2027E, matching the claim.",historical CAGR attributed to a forecast period
145,CAGR attributed to wrong period,AMD,competitor_analysis,step1,"Over the 2021-2027 period, AMD’s revenue is projected to grow at a strong 20.5% CAGR.","The table shows a 20.5% CAGR for AMD's revenue from 2021A to 2027E, matching the claim.",historical CAGR attributed to a forecast period
125,CAGR attributed to wrong period,AMD,investment_overview,step1,"With a projected revenue CAGR of 20.5% through 2027, margin expansion, and earnings recovery, AMD is well positioned to benefit from sec...",The evidence shows a projected revenue CAGR of 20.5% from 2021A to 2027E.,historical CAGR attributed to a forecast period
90,CAGR attributed to wrong period,AMD,tagline,step1,AMD demonstrates robust top-line growth with a projected revenue CAGR of 20.5% through 2027.,"The table shows a revenue CAGR of 20.5% from 2021A to 2027E, matching the claim.",historical CAGR attributed to a forecast period
134,CAGR attributed to wrong period,AMD,valuation_overview,step1,"AMD’s EBITDA growth outlook (CAGR of ~20% for revenue, strong margin expansion to 42% by 2027E) justifies some premium.","The revenue CAGR is 20.5% and the 2027E EBITDA margin is 42%, matching the claim.",historical CAGR attributed to a forecast period


## Cross-checks on the judge

*Rule overrides*: claims the judge called Supported/Unsupported that a rule turned into Contradicted. *Number check*:
for claims containing $, % or x figures, whether every figure can be found in the evidence (or derived from it).
A Supported claim with a figure that cannot be found, or a Contradicted claim whose figures all exist, is not
necessarily wrong — period mislabels keep correct numbers — but these are the claims worth reading first.

In [9]:
print(f"rules fired on {(claims.rule_flags != '').sum()} of {len(claims)} claims and changed the verdict on "
      f"{(claims.judge_verdict != claims.verdict).sum()}")
print(pd.crosstab(claims.judge_verdict, claims.verdict, margins=True), "\n")
numeric = claims[claims.numbers_total > 0].assign(
    numbers=lambda d: (d.numbers_found == d.numbers_total).map({True: "all found", False: "some not found"}))
print(pd.crosstab(numeric.verdict, numeric.numbers, margins=True))
print("\nquotes that are not exact substrings of the section:", (~claims.quote_in_text).sum())

rules fired on 41 of 438 claims and changed the verdict on 30
verdict        Contradicted  Supported  Unsupported  All
judge_verdict                                           
Contradicted             49          0            0   49
Supported                30        243            0  273
Unsupported               0          0          116  116
All                      79        243          116  438 

numbers       all found  some not found  All
verdict                                     
Contradicted         62               0   62
Supported           146               2  148
Unsupported           3               0    3
All                 211               2  213

quotes that are not exact substrings of the section: 25


## Hand review of a sample

`audit/grounding_review.csv` holds a stratified sample — every rule override, every Supported claim whose numbers
could not all be found, and a random draw from each verdict — with a reviewer verdict for each. The judge's accuracy
on that sample is what the grounding rates above should be read against.

In [10]:
review_path = ROOT / "audit" / "grounding_review.csv"
if review_path.exists():
    review = pd.read_csv(review_path)
    done = review.dropna(subset=["reviewer_verdict"])
    merged = done.merge(claims[["claim_id", "verdict"]], on="claim_id", how="left", suffixes=("", "_now"))
    agree = (merged["verdict_now"] == merged["reviewer_verdict"]).mean()
    print(f"reviewed {len(done)} claims; scorer (judge + rules) agrees with the reviewer on {agree:.0%}")
    print(pd.crosstab(merged["verdict_now"].rename("scorer"), merged["reviewer_verdict"].rename("reviewer"), margins=True))
    disagreements = merged[merged["verdict_now"] != merged["reviewer_verdict"]]
    display(disagreements[["claim_id", "claim", "verdict_now", "reviewer_verdict", "reviewer_note"]])
    # Only the random stratum is representative: use it to correct the step-1 grounding rate for judge errors.
    rnd = merged[merged["stratum"] == "random"]
    p_supported = rnd.groupby("verdict_now")["reviewer_verdict"].apply(lambda s: (s == "Supported").mean())
    counts = claims[claims.variant == "step1"]["verdict"].value_counts()
    adjusted = sum(counts[v] * p_supported.get(v, 0) for v in counts.index) / counts.sum()
    print(f"step-1 grounding rate: measured {counts['Supported'] / counts.sum():.0%}; corrected with the judge's error "
          f"rates on the {len(rnd)} random claims: {adjusted:.0%} (small sample, indicative only)")
else:
    print("no review file yet")

reviewed 65 claims; scorer (judge + rules) agrees with the reviewer on 91%
reviewer      Contradicted  Supported  Unsupported  All
scorer                                                 
Contradicted            37          1            0   38
Supported                2         12            3   17
Unsupported              0          0           10   10
All                     39         13           13   65


,claim_id,claim,verdict_now,reviewer_verdict,reviewer_note
31,GOOGL-investment_overview-step1-13,"The stock trades at a forward PE of 23–28x, in line with sector averages",Supported,Unsupported,"The P/E range matches the forecast P/E (24.7-27.3x), but nothing in the evidence gives sector averages."
36,GOOGL-competitor_analysis-report-9,Meta has a lower EV/EBITDA ratio indicative of its current struggles.,Supported,Unsupported,A lower EV/EBITDA does not by itself establish 'current struggles'; Meta's EBITDA rose 37.7B -> 104.5B.
41,INTC-valuation_overview-step1-9,"Intel’s long-term EV/EBITDA (14-15x) appears reasonable, assuming successful execution of turnaround strategies.",Supported,Unsupported,"A valuation opinion; the only 14-15x data point is 2025 (history 6.5-106.9x), so 'long-term' is not established."
42,AMD-company_overview-step1-10,"EBITDA, a key operating metric, dipped in 2023 but is projected to more than double from $5.3 billion in 2024 to $8.6 billion in 2027, r...",Supported,Contradicted,+64% is not 'more than double'.
43,GOOGL-investment_overview-step1-2,Growth accelerating to 13.9% in 2024 (guidance: $350.0 billion),Supported,Contradicted,'Guidance' means management's forward outlook; $350.0B is the reported 2024A revenue and the evidence has no guidance.
58,GOOGL-investment_overview-step1-8,EBITDA grew 15% in 2023 to $97.97 billion,Contradicted,Supported,Wording is fine and 85.2B -> 98.0B is +15.0%.


step-1 grounding rate: measured 57%; corrected with the judge's error rates on the 33 random claims: 44% (small sample, indicative only)


## Findings

**Overall.** Across 345 claims in the Step 1 sections of five reports, 57% are supported by the prompt, 26% are
unsupported (nothing in the prompt establishes them) and 18% are contradicted by it. The split by claim type is the
key to reading this. Quantitative claims (211) are 75% supported and only 1% unsupported, but **23% are
contradicted**. Qualitative claims (134) are 28% supported and 63% unsupported. Put simply: when the model writes
numbers it uses the tables but often gets them wrong, and when it writes prose it relies on prior knowledge.

| Section (Step 1) | Claims | Grounding rate | Unsupported | Contradicted |
|---|---|---|---|---|
| company_overview | 82 | 39% | 55% | 6% |
| investment_overview | 76 | 51% | 26% | 22% |
| valuation_overview | 54 | 57% | 19% | 24% |
| tagline | 25 | 64% | 20% | 16% |
| risks | 36 | 75% | 6% | 19% |
| competitor_analysis | 52 | 79% | 12% | 10% |
| major_takeaways (canned fallback in 5/5 runs) | 20 | 50% | 0% | 50% |
| *in the report:* competitor_analysis (gpt-4o-mini) | 60 | 45% | 43% | 12% |
| *in the report:* major_takeaways (gpt-4o-mini) | 33 | 61% | 6% | 33% |

**Which sections hallucinate most, and why it is structural**

1. **Unsupported claims follow the gap between what the system prompt asks for and what the user prompt contains.**
   `company_overview` is told to cover "business model, products/services, market position" and is given three
   financial tables. It can only answer from memory, so 55% of its claims are unsupported (founding year, headquarters,
   product lines, "industry standard"). The report's `competitor_analysis`, regenerated by gpt-4o-mini, does the same
   for market positions (43% unsupported, against 12% for the gpt-4.1 Step 1 version it replaces).
2. **Contradictions come from the evidence format, not from the model inventing numbers.** Two thirds of the 79
   contradicted claims are period or status errors: a reported value presented as a projection, estimate or
   "guidance" (31), or the historical CAGR presented as a forecast "through 2027" (22). The prompt invites both. The
   forecast columns are labelled `2025E–2027E` and sit next to `2025A`/`2026A`; the CAGR column does not say which
   years it covers; the peer tables do not mark actual vs. projected; and AMD's 2027 EBITDA appears twice with
   different values ($16.8bn in the metrics table, $8.6bn in the peer projection). Sections that ask for forward-looking
   language ("outlook", "fair value assessment", a one-line thesis) have the highest contradiction rates:
   investment_overview 22%, valuation_overview 24%, tagline 16%.
3. **The fallback logic manufactures claims.** `major_takeaways` was replaced by canned text in all five runs. That text
   asserts "consistent performance trends" and "margin stability" for every company, and for Intel (revenue −20%,
   −14%, −2%; EBITDA margin 42.9% → 2.3%) three of its four sentences are contradicted by the data.
4. **The assignment's hypothesis about `risks` does not apply to the code that runs.** The 600–800-word risks agent is
   never called; the CLI asks for "5 key investment risks, concise". The model anchors each bullet on a table number
   and phrases everything else as hypotheticals ("could erode margins if..."). Hypotheticals are not checkable claims,
   so the section scores well (75%). It is still not risk analysis: NVIDIA's five risks are restatements of valuation
   and growth figures (two of them wrong), wrapped in "Certainly. Here are..." and "Let me know if you need...".
   Regulation, supply chain and customer concentration can only appear as generic hypotheticals, because no data on
   them reaches the prompt.

**How far to trust these numbers.** On a hand-reviewed sample of 65 claims, the scorer (judge + rules) agrees with the
reviewer on 91%. All 30 rule overrides were right. On the 33 randomly drawn claims agreement is 85%, and the judge
errs towards leniency: 4 of 15 random "Supported" claims should not have been ("more than double" for +64%, "guidance"
for a reported figure, "current struggles" the data does not show, a "long-term" multiple seen in one year). Corrected for those error rates, the Step 1
grounding rate is about 44% rather than 57%. The sample is small, so treat the correction as indicative. Claude reviewed all 65 claims first; the six on which that
review disagreed with the scorer were then adjudicated independently by Yuxi Luo, who reached the same verdict on all
six. The 59 agreements were reviewed once.

**Not covered here.** `news_summary` (no news on the FMP free plan, so Step 1 writes a placeholder); run-to-run
stability is in `2_3b_stability.ipynb`.
